In [3]:
from torch.utils.data import DataLoader
import pickle 

from codefiles.datasets.mimic_symile import MIMIC_Symile
from codefiles.datasets.mimic_haim import MIMIC_Haim
from codefiles.datasets.mosi_mosei import MOSI_MOSEI
from codefiles.datasets.ch_sims import CH_Sims, collate_fn
from codefiles.datasets.ch_sims_v2 import CH_Sims_v2, collate_fn_v2
from codefiles.datasets.crema_d import CREMAD
from codefiles.datasets.vgg_sound import VGGSound
from codefiles.datasets.vision_touch import VisionTouch
from codefiles.datasets.kinetics import Kinetics
from codefiles.datasets.inspect import INSPECT, inspect_collate_fn

train_ds = MOSI_MOSEI(dataset="mosi", split_nr=0)

print(len(train_ds))

total_samples: 1319 / 1319
no_missing: 1319 / 1319
1_missing: 0 / 1319
2_missing: 0 / 1319
modality_0_missing: 0 / 1319
modality_1_missing: 0 / 1319
modality_2_missing: 0 / 1319
1319


In [11]:
import pandas as pd 
import itertools

mosi_path_split1 = f"/sc-projects/sc-proj-ukb-cvd/projects/data/MOSEI/aligned_50_testsplits_split_0.pkl"
mosi_path_split2 = f"/sc-projects/sc-proj-ukb-cvd/projects/data/MOSEI/aligned_50_testsplits_split_1.pkl"
mosi_path_split3 = f"/sc-projects/sc-proj-ukb-cvd/projects/data/MOSEI/aligned_50_testsplits_split_2.pkl"
mosi_path_split4 = f"/sc-projects/sc-proj-ukb-cvd/projects/data/MOSEI/aligned_50_testsplits_split_3.pkl"
mosi_path_split5 = f"/sc-projects/sc-proj-ukb-cvd/projects/data/MOSEI/aligned_50_testsplits_split_4.pkl"

mosi_split1 = pd.read_pickle(mosi_path_split1)
mosi_split2 = pd.read_pickle(mosi_path_split2)
mosi_split3 = pd.read_pickle(mosi_path_split3)
mosi_split4 = pd.read_pickle(mosi_path_split4)
mosi_split5 = pd.read_pickle(mosi_path_split5)

print(len(mosi_split1["test"]["id"]))

splits = {
    1: mosi_split1,
    2: mosi_split2,
    3: mosi_split3,
    4: mosi_split4,
    5: mosi_split5
}

# Extract test IDs for each split and convert to sets
test_id_sets = {num: set(split["test"]["id"]) for num, split in splits.items()}

print("Checking for overlaps in test IDs between all splits...")

# Get all unique pairs of split numbers
split_numbers = list(splits.keys())
for i, j in itertools.combinations(split_numbers, 2):
    set1 = test_id_sets[i]
    set2 = test_id_sets[j]

    # Find the intersection
    overlap = set1.intersection(set2)

    if overlap:
        print(f"Found {len(overlap)} overlapping IDs between split {i} and split {j}:")
        # The line below will print all overlapping IDs.
        # You can comment it out if the list is too long.
        print(f"  {overlap}")
    else:
        print(f"No overlap between split {i} and split {j}.")

4572
Checking for overlaps in test IDs between all splits...
No overlap between split 1 and split 2.
No overlap between split 1 and split 3.
No overlap between split 1 and split 4.
No overlap between split 1 and split 5.
No overlap between split 2 and split 3.
No overlap between split 2 and split 4.
No overlap between split 2 and split 5.
No overlap between split 3 and split 4.
No overlap between split 3 and split 5.
No overlap between split 4 and split 5.


In [1]:
import torch

torch.cumsum(torch.tensor([0] + [14, 51]), dim=0)

tensor([ 0, 14, 65])

In [8]:
import torch 

split = "train"
split_nr = 2

dataset_path = "/sc-resources/dh-mimic/mimic_symile/mimic_symile/"
cxrs = torch.load(f"{dataset_path}/{split}/cxr_{split}{split_nr}.pt")

cxrs[0].sum()


tensor(18887.2793)

In [12]:
import pandas as pd 
import numpy as np

dataset_path = f"/sc-projects/sc-proj-ukb-cvd/projects/data/datasets--Loie--VGGSound/blobs/vggsound_fullfilename_splits.csv"
dataset_csv = pd.read_csv(dataset_path)

# Get the total number of samples
num_samples = len(dataset_csv)
indices = np.arange(num_samples)

# Shuffle indices for random fold creation
np.random.seed(42) # for reproducibility
np.random.shuffle(indices)

# Split indices into 5 folds
n_folds = 5
fold_indices = np.array_split(indices, n_folds)

# Create new split columns for each of the 5 folds
for i in range(n_folds):
    # Determine which folds are for train, val, and test
    test_fold_idx = i
    val_fold_idx = (i + 1) % n_folds
    
    # Get the actual sample indices for each set
    test_indices = fold_indices[test_fold_idx]
    val_indices = fold_indices[val_fold_idx]
    
    # Create the new split column and assign splits
    new_col_name = f'cv_split_{i}'
    # Initialize column with 'train' as the default
    dataset_csv[new_col_name] = 'train'
    # Set validation and test sets using .loc for safe assignment
    dataset_csv.loc[val_indices, new_col_name] = 'valid'
    dataset_csv.loc[test_indices, new_col_name] = 'test'

# --- Verification ---
print("New cross-validation splits created as 'cv_split_0' to 'cv_split_4'.\n")

# Check the distribution of one of the new splits
print("Value counts for 'cv_split_0':")
print(dataset_csv['cv_split_0'].value_counts())
print("-" * 30)

# Verify that the test sets are indeed different across splits
test_set_0_indices = set(dataset_csv.index[dataset_csv['cv_split_0'] == 'test'])
test_set_1_indices = set(dataset_csv.index[dataset_csv['cv_split_1'] == 'test'])
overlap = test_set_0_indices.intersection(test_set_1_indices)

if not overlap:
    print("Test sets for cv_split_0 and cv_split_1 are disjoint, as expected for CV.")
else:
    print(f"Warning: Test sets for cv_split_0 and cv_split_1 overlap by {len(overlap)} samples.")

print("\nShowing original and new splits for comparison:")
dataset_csv[[c for c in dataset_csv.columns if 'split' in c]].head()

New cross-validation splits created as 'cv_split_0' to 'cv_split_4'.

Value counts for 'cv_split_0':
cv_split_0
train    119679
valid     39894
test      39894
Name: count, dtype: int64
------------------------------
Test sets for cv_split_0 and cv_split_1 are disjoint, as expected for CV.

Showing original and new splits for comparison:


,split,split_0,split_1,split_2,split_3,split_4,cv_split_0,cv_split_1,cv_split_2,cv_split_3,cv_split_4
0,train,val,train,train,train,train,valid,test,train,train,train
1,train,train,train,train,train,val,train,valid,test,train,train
2,train,val,train,train,train,train,train,train,valid,test,train
3,train,train,train,train,train,val,train,valid,test,train,train
4,train,val,train,train,train,train,test,train,train,train,valid


In [17]:
dataset_csv.to_csv(f"/sc-projects/sc-proj-ukb-cvd/projects/data/kinetics/kinetics-dataset/k400/annotations/kinetics_trainval_cv_splits.csv", index=False)

In [106]:
import torch
import glob
import numpy as np
import pandas as pd
from torch.utils.data import Dataset

import tqdm

class INSPECT(Dataset):
    def __init__(
        self, 
        dataset_path: str = "/sc-projects/sc-proj-ukb-cvd/projects/data/inspect/",
        split: str = "train",
        split_nr: int = 1, 
        variant: str = "unimodal_1",
        zero_fill_rates: list = [0.0],
        seed: int = 42
    ) -> None: 
        super().__init__()

        self.num_modalities = 2
        self.variant = variant

        splits_df = pd.read_csv(f"/sc-projects/sc-proj-ukb-cvd/projects/25_multimodal_llm/INSPECT/data/INSPECT20250611/splits_20250611.tsv", sep="\t")
        splits_df = splits_df[splits_df["split"] == split]

        self.targets_df = pd.read_pickle(dataset_path + "PE_targets.pkl")
        self.targets_df["label"] = self.targets_df["label"].apply(lambda x: False if x == "False" else True)
        self.targets_df["label"] = self.targets_df["label"].astype(bool)
        self.ehr_df = pd.read_pickle(dataset_path + "ehr_motor_embeddings.pkl")
        self.images_pathlist = glob.glob(dataset_path + "vision_radfm_embeddings/*.npz")

        # filter for split 
        self.targets_df = self.targets_df[self.targets_df["patient_id"].isin(splits_df["person_id"])]
        self.ehr_df = self.ehr_df[self.ehr_df["patient_ids"].isin(splits_df["person_id"])]
        self.images_pathlist = [file for file in self.images_pathlist if file.split("/")[-1].split(".")[0] in self.ehr_df["image_id"].tolist()]

        # still on a subset of the data
        #self.image_ids = self.targets_df["image_id"].tolist()
        self.image_ids = [id.split(".")[0].split("/")[-1] for id in self.images_pathlist]

        # missing data
        #self.zero_fill_masks, self.missing_stats = create_missing_data_masks(
        #    total_samples=len(self.haim_dataset),
        #    num_modalities=self.num_modalities,
        #    missing_rates=zero_fill_rates,
        #    random_seed=seed
        #)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]

        # Image
        image_path = f"/sc-projects/sc-proj-ukb-cvd/projects/data/inspect/vision_radfm_embeddings/{image_id}.npz"
        image = np.load(image_path)["layer_0"]
        image = torch.from_numpy(image).float()

        # EHR
        ehr = self.ehr_df.loc[self.ehr_df["image_id"] == image_id, "embeddings"].values[0]
        ehr = torch.tensor(ehr)

        # Target
        target = self.targets_df.loc[self.targets_df["image_id"] == image_id, "label"].values[0]
        target = torch.from_numpy(np.array(target)).float().unsqueeze(0)

        # full multimodal datareturn
        datareturn = [image, ehr, target]

        # Missing Data
        #if not "unimodal" in self.variant:
        #    datareturn[0] = apply_missing_mask(datareturn[0], self.zero_fill_masks[idx, 0])
        #    datareturn[1] = apply_missing_mask(datareturn[1], self.zero_fill_masks[idx, 1])

        return {
            "image": datareturn[0], 
            "ehr": datareturn[1], 
            "target": target
        }

inspect_dataset = INSPECT(split = "valid")
inspect_dataloader = DataLoader(inspect_dataset, batch_size=1, shuffle=False)
for batch in tqdm.tqdm(inspect_dataloader):
    pass 


100%|██████████| 400/400 [00:01<00:00, 322.60it/s]


In [ ]:
import pickle
import pandas as pd 
import torch

inspect_dir = f"/sc-projects/sc-proj-ukb-cvd/projects/data/inspect/"

targets_df = pd.read_pickle(inspect_dir + "PE_targets.pkl")
ehr_df = pd.read_pickle(inspect_dir + "ehr_motor_embeddings.pkl")
images_pathlist = glob.glob(inspect_dir + "vision_radfm_embeddings/*.npz")

display(ehr_df)
ehr = ehr_df.loc[ehr_df["image_id"] == "PE45272eb", "embeddings"].values[0]
ehr = torch.tensor(ehr)


,embeddings,patient_ids,prediction_time,image_id,impression_id
0,"[-1.0791016, 2.9863281, -0.67089844, 0.1061401...",115967101,2020-07-20 10:59:00,PE45272eb,22024
1,"[-1.7402344, 2.125, 0.47265625, -0.5830078, 1....",115967141,2017-06-29 15:44:00,PE4529a1f,311
2,"[-0.12817383, 2.6914062, 0.9238281, 0.68359375...",115967149,2020-10-17 21:13:00,PE4526cf6,20044
3,"[-0.18444824, 2.6953125, 0.61035156, -0.336425...",115967149,2021-03-30 18:08:00,PE452429e,20043
4,"[-1.5771484, 2.6308594, -0.34985352, -0.937011...",115967152,2019-05-30 12:21:00,PE4527e1b,11543
...,...,...,...,...,...
22452,"[-0.24511719, 1.9228516, -0.50683594, -0.29467...",128560398,2021-01-27 03:08:00,PE4528dca,3340
22453,"[0.8676758, 0.9916992, 0.7241211, -0.22094727,...",128560402,2012-09-28 03:26:00,PE87ed00,11576
22454,"[-0.8041992, 1.7910156, -0.7788086, -0.9663086...",128560598,2015-08-18 13:35:00,PE9f4d71,12851
22455,"[-1.5546875, 2.3496094, 0.1484375, -0.7583008,...",128560927,2020-01-11 07:26:00,PE452941f,13058


torch.float32

In [ ]:
import pickle
import pandas as pd 

data_path = f"/sc-projects/sc-proj-ukb-cvd/projects/ctrate/INSPECT_public/ehr/pkl_files_ben/labels_and_features/PE/featurized_patients_with_image_ids.pkl"

data = pd.read_pickle(data_path)

print(data)


(<22457x82112 sparse matrix of type '<class 'numpy.float32'>'
	with 11750532 stored elements in Compressed Sparse Row format>, array([115967101, 115967141, 115967149, ..., 128560598, 128560927,
       128729714]), array(['False', 'False', 'False', ..., 'False', 'False', 'False'],
      dtype='<U5'), array(['2020-07-20T10:59:00.000000', '2017-06-29T15:44:00.000000',
       '2020-10-17T21:13:00.000000', ..., '2015-08-18T13:35:00.000000',
       '2020-01-11T07:26:00.000000', '2020-10-10T15:02:00.000000'],
      dtype='datetime64[us]'), array(['PE45272eb', 'PE4529a1f', 'PE4526cf6', ..., 'PE9f4d71',
       'PE452941f', 'PE452afa5'], dtype=object), array([22024,   311, 20044, ..., 12851, 13058,  2590]))


In [111]:
import pandas as pd 

motor_path = f"/sc-projects/sc-proj-ukb-cvd/projects/ctrate/INSPECT_public/ehr/pkl_files_ben/labels_and_features/PE/featurized_patients_with_image_ids.pkl"

data = pd.read_pickle(motor_path)

# The data is a tuple containing a sparse matrix and several arrays.
# Let's unpack it to inspect each part. Based on the output, we can guess the contents:
(
    features_matrix,
    patient_ids,
    labels,
    timestamps,
    image_ids,
    some_other_ids,
) = data

# First, let's print some info about the large sparse feature matrix
print("--- Feature Matrix ---")
print(f"Type: {type(features_matrix)}")
print(f"Shape: {features_matrix.shape}")
print(f"Stored elements: {features_matrix.nnz}")
density = (features_matrix.nnz / (features_matrix.shape[0] * features_matrix.shape[1])) * 100
print(f"Density: {density:.4f}%\n")

# show features_matrix 
print(features_matrix)


# For the other arrays (metadata, labels, etc.), a pandas DataFrame is perfect for viewing.
print("--- Metadata and Labels ---")
df = pd.DataFrame({
    "patient_id": patient_ids,
    "label": labels,
    "timestamp": timestamps,
    "image_id": image_ids,
    "other_id": some_other_ids,
})

# In a Jupyter notebook, simply having the DataFrame as the last line of a cell
# will render it as a nice-looking table. This shows the first 5 rows.
df.head()

# safe df as pkl
# df.to_pickle(f"/sc-projects/sc-proj-ukb-cvd/projects/data/inspect/PE_targets.pkl")

--- Feature Matrix ---
Type: <class 'scipy.sparse._csr.csr_matrix'>
Shape: (22457, 82112)
Stored elements: 11750532
Density: 0.6372%

  (np.int32(0), np.int32(0))	0.6121606826782227
  (np.int32(0), np.int32(63141))	1.0
  (np.int32(0), np.int32(5521))	1.0
  (np.int32(0), np.int32(79956))	1.0
  (np.int32(0), np.int32(80392))	1.0
  (np.int32(0), np.int32(80641))	1.0
  (np.int32(0), np.int32(79816))	159.0
  (np.int32(0), np.int32(79855))	1.0
  (np.int32(0), np.int32(80625))	56.0
  (np.int32(0), np.int32(5520))	1.0
  (np.int32(0), np.int32(48114))	1.0
  (np.int32(0), np.int32(48103))	1.0
  (np.int32(0), np.int32(48110))	1.0
  (np.int32(0), np.int32(50046))	1.0
  (np.int32(0), np.int32(50029))	1.0
  (np.int32(0), np.int32(50045))	1.0
  (np.int32(0), np.int32(47103))	1.0
  (np.int32(0), np.int32(47100))	1.0
  (np.int32(0), np.int32(47868))	1.0
  (np.int32(0), np.int32(47858))	1.0
  (np.int32(0), np.int32(47865))	1.0
  (np.int32(0), np.int32(51750))	1.0
  (np.int32(0), np.int32(51748))	1.0
  (

,patient_id,label,timestamp,image_id,other_id
0,115967101,False,2020-07-20 10:59:00,PE45272eb,22024
1,115967141,False,2017-06-29 15:44:00,PE4529a1f,311
2,115967149,False,2020-10-17 21:13:00,PE4526cf6,20044
3,115967149,False,2021-03-30 18:08:00,PE452429e,20043
4,115967152,False,2019-05-30 12:21:00,PE4527e1b,11543


In [31]:
import glob
import numpy as np
import tqdm

radfm_embeddings_dir = f"/sc-scratch/sc-scratch-dh-fu-swp-25/inspect/inspect2/radfm_embeddings_mean_PE/"

npz_files = glob.glob(radfm_embeddings_dir + "*.npz")

# filter npz_files to only include files that contain "_v_"
npz_files = [file for file in npz_files if "_v_" in file]

# save every npz file in npz_files under /sc-projects/sc-proj-ukb-cvd/projects/data/inspect/vision_radfm_embeddings/
for file in tqdm.tqdm(npz_files):
    data = np.load(file)
    filename = file.split('/')[-1]
    image_id = filename.split('_radfm_v_')[0]
    np.savez_compressed(f"/sc-projects/sc-proj-ukb-cvd/projects/data/inspect/vision_radfm_embeddings/{image_id}.npz", **data)



100%|██████████| 8672/8672 [01:20<00:00, 108.37it/s]


In [108]:
# open /sc-projects/sc-proj-ukb-cvd/projects/25_multimodal_llm/INSPECT/data/INSPECT20250611/splits_20250611.tsv

splits_df = pd.read_csv(f"/sc-projects/sc-proj-ukb-cvd/projects/25_multimodal_llm/INSPECT/data/INSPECT20250611/splits_20250611.tsv", sep="\t")

display(splits_df)

from sklearn.model_selection import KFold

# 1. Identify the unique person_ids from the original training set
train_person_ids = splits_df[splits_df['split'] == 'train']['person_id'].unique()

# 2. Create a mapping from each training person_id to a fold number (0-4)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
person_id_to_fold = {}
for fold_idx, (_, val_indices) in enumerate(kf.split(train_person_ids)):
    val_person_ids = train_person_ids[val_indices]
    for person_id in val_person_ids:
        person_id_to_fold[person_id] = fold_idx

# 3. Create a temporary 'fold' column to simplify the logic.
# Patients in original 'valid'/'test' sets get -1.
splits_df['fold'] = splits_df['person_id'].map(person_id_to_fold).fillna(-1).astype(int)

# 4. Generate the five new split columns
for k in range(5):
    col_name = f'split_{k}'
    
    # Start by copying the original split assignments. This correctly sets 'test'
    # and handles the original 'valid' set if you need to keep it separate.
    splits_df[col_name] = splits_df['split']
    
    # For patients from the original training pool, re-assign their status.
    # If a person's assigned fold is k, they are 'valid' for split_k.
    is_valid_for_fold_k = (splits_df['fold'] == k)
    
    # If they are in any other fold, they are 'train' for split_k.
    is_train_for_fold_k = (splits_df['fold'] != k) & (splits_df['fold'] != -1)
    
    # Use .loc for safe assignment based on the conditions
    splits_df.loc[is_valid_for_fold_k, col_name] = 'valid'
    splits_df.loc[is_train_for_fold_k, col_name] = 'train'

# Optional: Drop the temporary 'fold' column
splits_df = splits_df.drop(columns=['fold'])


# --- Verification ---
# Check the distribution of one of the new splits.
# The 'valid' count should be ~1/5th of the original 'train' count.
# The 'train' count should be ~4/5ths of the original 'train' count.
print(f"--- Value counts for split_0 ---\n{splits_df['split_0'].value_counts()}\n")

# Pick a random person from the original training set and inspect their new assignments
example_person_id = train_person_ids[10]
print(f"--- Assignments for person_id {example_person_id} ---")
display(splits_df[splits_df['person_id'] == example_person_id][['person_id', 'split_0', 'split_1', 'split_2', 'split_3', 'split_4']].head(1))
print("\n" + "="*50 + "\n")


print("--- Final DataFrame Head ---")
display(splits_df.head())

# save splits_df as pkl under /sc-projects/sc-proj-ukb-cvd/projects/data/inspect/cv_splits.pkl
splits_df.to_pickle(f"/sc-projects/sc-proj-ukb-cvd/projects/data/inspect/cv_splits.pkl")

,impression_id,person_id,split
0,0,125058905,train
1,1,126763689,valid
2,2,125316107,test
3,3,125316107,test
4,4,125316107,test
...,...,...,...
23243,23243,126525462,train
23244,23244,125928187,train
23245,23245,126274542,test
23246,23246,127290280,test


--- Value counts for split_0 ---
split_0
train    15098
valid     4936
test      3214
Name: count, dtype: int64

--- Assignments for person_id 126386010 ---


,person_id,split_0,split_1,split_2,split_3,split_4
19,126386010,train,valid,train,train,train




--- Final DataFrame Head ---


,impression_id,person_id,split,split_0,split_1,split_2,split_3,split_4
0,0,125058905,train,valid,train,train,train,train
1,1,126763689,valid,valid,valid,valid,valid,valid
2,2,125316107,test,test,test,test,test,test
3,3,125316107,test,test,test,test,test,test
4,4,125316107,test,test,test,test,test,test
